<a href="https://colab.research.google.com/github/Anna94652/10701-Secondary-Protein-Structure-Prediction/blob/main/701ProjectDataPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [77]:
!pip install huggingface_hub -q
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np

In [78]:
# Training set
train_path = hf_hub_download(
    repo_id="proteinea/secondary_structure_prediction",
    filename="training_hhblits.csv",
    repo_type="dataset"
)
train_df = pd.read_csv(train_path)
print(train_df.shape)
print(train_df.columns.tolist())

# Test set (CB513)
test_path = hf_hub_download(
    repo_id="proteinea/secondary_structure_prediction",
    filename="CB513.csv",
    repo_type="dataset"
)
test_df = pd.read_csv(test_path)
print(test_df.shape)

(10792, 5)
['input', 'dssp3', 'dssp8', 'disorder', 'cb513_mask']
(511, 5)


In [ ]:
# See all unique characters actually present in the labels
all_chars = set(''.join(train_df['dssp8'].dropna()))
print(sorted(all_chars))

In [ ]:
# Amino acid vocabulary
AA_VOCAB_str = 'ACDEFGHIKLMNPQRSTVWXY'
SS_VOCAB_str  = 'CEHT'     # dssp3 (3 classes)
SS8_VOCAB_str = 'CBEGIHST' # dssp8 (8 classes)

MAX_LEN = 700  # pad/truncate all sequences to this length

In [ ]:
def encode_sequence(seq, vocab, max_len):
    vocab_map = {c: i for i, c in enumerate(vocab)}
    n = len(vocab)
    arr = np.zeros((max_len, n), dtype=np.float32)
    for i, ch in enumerate(seq[:max_len]):
        if ch in vocab_map:
            arr[i, vocab_map[ch]] = 1.0
    return arr   # shape (max_len, n)

In [ ]:
def encode_df(df):
    X, y = [], []
    for _, row in df.iterrows():
        X.append(encode_sequence(row['input'], AA_VOCAB, MAX_LEN))
        y.append(encode_sequence(row['dssp8'], 'CBEGIHST', MAX_LEN))
    return np.array(X), np.array(y)

X_train, y_train = encode_df(train_df)
X_test,  y_test  = encode_df(test_df)

print(X_train.shape)  # (N, 700, 21)
print(y_train.shape)  # (N, 700, 3)

In [ ]:
def make_mask(df, max_len=MAX_LEN):
    masks = []
    for _, row in df.iterrows():
        length = min(len(row['input']), max_len)
        mask = np.zeros(max_len, dtype=np.float32)
        mask[:length] = 1.0
        masks.append(mask)
    return np.array(masks)   # shape (N, 700)

train_mask = make_mask(train_df)
test_mask  = make_mask(test_df)

In [ ]:
AA_VOCAB = list(AA_VOCAB_str)
SS_VOCAB = list(SS_VOCAB_str)
SS8_VOCAB = list(SS8_VOCAB_str)

def view_protein(X, y, idx):
    # Find where the sequence ends (all zeros = padding)
    length = int((X[idx].sum(axis=1) > 0).sum())

    aa_df  = pd.DataFrame(X[idx, :length, :], columns=AA_VOCAB)
    ss_df  = pd.DataFrame(y[idx, :length, :], columns=SS8_VOCAB)

    # Decode back to letters for readability
    aa_df['amino_acid'] = aa_df.idxmax(axis=1)
    ss_df['structure']  = ss_df.idxmax(axis=1)

    return pd.concat([aa_df[['amino_acid']], ss_df[['structure']]], axis=1)

view_protein(X_train, y_train, 10791)

In [ ]:
# One row per protein — sequence length and most common structure
SS8_VOCAB = ['C', 'B', 'E', 'G', 'I', 'H', 'S', 'T']

mask = (X_train.sum(axis=2) > 0)  # (N, 700) — True where real residue

summary = pd.DataFrame(
    {f'pct_{ss}': (y_train[:, :, i] * mask).sum(axis=1) / mask.sum(axis=1)
     for i, ss in enumerate(SS8_VOCAB)},
)
summary.insert(0, 'length', mask.sum(axis=1))

print(summary.describe())

In [ ]:
print(train_df.isnull().sum())
print(train_df['input'].str.len().describe())  # sequence length distribution

In [ ]:
train_sequences = set(train_df['input'].str.strip())
test_sequences  = set(test_df['input'].str.strip())

overlap = train_sequences & test_sequences
print(f"Training proteins:       {len(train_sequences)}")
print(f"Test proteins (CB513):   {len(test_sequences)}")
print(f"Overlapping sequences:   {len(overlap)}")

In [ ]:
train_df_clean = train_df[~train_df['input'].str.strip().isin(test_sequences)]
print(f"Training set after removing overlap: {len(train_df_clean)}")

In [ ]:
# Normalise fully before comparing
train_df['input_clean'] = train_df['input'].str.strip().str.upper()
test_df['input_clean']  = test_df['input'].str.strip().str.upper()

overlap = set(train_df['input_clean']) & set(test_df['input_clean'])
print(f"Overlap after normalisation: {len(overlap)}")
train_df.drop(columns=['input_clean'], inplace=True)
test_df.drop(columns=['input_clean'], inplace=True)

In [ ]:
100* (len(test_df) /len(train_df))

In [ ]:
train_df.head()

In [ ]:
print(len(train_df['input'][0]))
print(train_df['input'][0])

In [ ]:
print(train_df['dssp8'][0])
print(len(train_df['dssp8'][0]))